In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    #non-perturbations
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets

    #perturbations
    #Run One: python Tracked_Profiles.py Variables_Perturbations
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracked_Profiles", dataName="Tracked_Profiles",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#JOB ARRAY

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def MakeDataDictionary(variableNames,t,printstatement=False):
    timeString = ModelData.timeStrings[t]
    # print(f"Getting data from {timeString}","\n")
    
    dataDictionary = {variableName: CallLagrangianArray(ModelData, DataManager, timeString, variableName=variableName, printstatement=printstatement) 
                      for variableName in variableNames}      
    return dataDictionary
    
def GetSpatialData(t):    
    variableNames = ['Z']
    dataDictionary = MakeDataDictionary(variableNames,t)
    [Z] = (dataDictionary[k] for k in variableNames)
    return Z

In [ ]:
####################################
#RUN SETUP

In [ ]:
#data variable list
def GetVarNames(dataName): 
    Processed_string = "PROCESSED_" if "PROCESSED_" in dataName else ""
    DivideMassFlux_string = "_DivideMassFlux" if "_DivideMassFlux" in dataName else ""

    
    if dataName=="Variables":
        varNames = ['W', 'QV','QCQI', 'RH_vapor', 'THETA_v', 'THETA_e', 'MSE', 'HMC', 'CONVERGENCE','VMF_g','VMF_c']
    elif dataName == "Entrainment":
        varNames = ['D_c', 'D_g', 'E_c', 'E_g',
                          'TransferD_c', 'TransferD_g', 
                          'TransferE_c', 'TransferE_g']
    elif dataName == f"{Processed_string}Entrainment{DivideMassFlux_string}":
        varNames = [f'{Processed_string}D{DivideMassFlux_string}_c', f'{Processed_string}D{DivideMassFlux_string}_g', 
                    f'{Processed_string}E{DivideMassFlux_string}_c', f'{Processed_string}E{DivideMassFlux_string}_g',
                    f'{Processed_string}TransferD{DivideMassFlux_string}_c', f'{Processed_string}TransferD{DivideMassFlux_string}_g', 
                    f'{Processed_string}TransferE{DivideMassFlux_string}_c', f'{Processed_string}TransferE{DivideMassFlux_string}_g']
    elif dataName=="W_Budgets":
        varNames = ['WB_BUOY', 'WB_HADV', 'WB_HIDIFF', 'WB_HTURB',
                    'WB_PGRAD', 'WB_VADV', 'WB_VIDIFF', 'WB_VTURB']
    elif dataName=="QV_Budgets":
        varNames = ['QVB_HADV', 'QVB_HIDIFF', 'QVB_HTURB', 'QVB_MP',
                    'QVB_VADV', 'QVB_VIDIFF', 'QVB_VTURB']
    elif dataName=="TH_Budgets":
        varNames = ['PTB_DISS', 'PTB_DIV', 'PTB_HADV', 'PTB_HIDIFF', 
                    'PTB_HTURB', 'PTB_MP', 'PTB_RAD', 'PTB_VADV', 
                    'PTB_VIDIFF', 'PTB_VTURB']

    elif dataName=="Variables_Perturbations":
        varNames = ["QV_prime","QCQI_prime","RH_vapor_prime",
                    "W_prime","VMF_g_prime",'VMF_c_prime',
                    "HMC_prime",
                    "THETA_v_prime",'THETA_e_prime','MSE_prime']
    return varNames

In [ ]:
########################################
#RUNNING FUNCTIONS

In [ ]:
#Functions for Initializing Profile Arrays
def CopyStructure(dictionary, placeholder=None):
    """Deep-copy dictionary structure, replacing leaves with a given placeholder."""
    if isinstance(dictionary, dict):
        return {k: CopyStructure(v, placeholder) for k, v in dictionary.items()}
    else:
        return placeholder

def InitializeProfileArrays(trackedArrays, varNames, zhs=ModelData.zh):
    """
    Create a new dictionary with the same nested structure as trackedArrays,
    and for each variable name, create:
        - 'profile_array' / 'profile_array_squares'
        - 'profile_array_left' / 'profile_array_left_squares'
        - 'profile_array_right' / 'profile_array_right_squares'
    Each array has shape (len(zhs), 3) and zhs in the last column.
    """
    profileArraysDictionary = {}

    for category, depth_dict in trackedArrays.items():  # e.g. 'CL', 'SBF'
        profileArraysDictionary[category] = {}

        for depth_type in depth_dict.keys():  # e.g. 'ALL', 'SHALLOW', 'DEEP'
            profileArraysDictionary[category][depth_type] = {}

            for varName in varNames:
                # Create base profile array
                base_profile = np.zeros((len(zhs), 3))
                base_profile[:, 2] = zhs

                profileArraysDictionary[category][depth_type][varName] = {
                    # Main / all parcels
                    "profile_array": base_profile.copy(),
                    "profile_array_squares": base_profile.copy(),

                    # Left subset (-1)
                    "profile_array_left": base_profile.copy(),
                    "profile_array_left_squares": base_profile.copy(),

                    # Right subset (+1)
                    "profile_array_right": base_profile.copy(),
                    "profile_array_right_squares": base_profile.copy(),
                }
    return profileArraysDictionary

In [ ]:
def GetParcelNumbers(trackedArray, t):
    """
    Return all parcel indices (p) and their corresponding row indices
    for parcels that are active at time t.
    Vectorized, no row-by-row loops.
    """
    t_start = trackedArray[:, 1]
    t_end   = np.minimum(trackedArray[:, 2] + trackedArray[:, 3], ModelData.Ntime)

    # Boolean mask for rows active at time t
    mask = (t >= t_start) & (t <= t_end)

    # Extract parcel numbers and their corresponding row indices
    selectedRows = np.where(mask)[0]
    selectedPs = trackedArray[selectedRows, 0]
    leftRightIndexes = trackedArray[selectedRows, 4]

    return selectedRows, selectedPs, leftRightIndexes

In [ ]:
def MakeTrackedProfiles(trackedArrays,profileArraysDictionary,varNames,VARs,Z,t):
    """
    Update profileArraysDictionary with variable data for parcels active at time t.
    Accumulates sums and counts in both profile_array and profile_array_squares.
    """
    #CALCULATING
    for key1, subdict in trackedArrays.items():         # e.g. 'CL', 'SBF'
        # print("\t",f'working on {key1}')
        for key2, trackedArray in subdict.items():           # e.g. 'ALL', 'DEEP'
            # print("\t\t",f'working on {key2}')
            #loading the profile array to fill
            profileArray = profileArraysDictionary[key1][key2] 
    
            #getting parcels in trackedArray to run through
            _, selectedPs, leftRightIndexes = GetParcelNumbers(trackedArray, t) #get parcels that are counted at time t
            
            #getting Z data
            zLevels = Z[selectedPs]

            # Boolean masks
            mask_left = leftRightIndexes == -1
            mask_right = leftRightIndexes == 1
            
            for varName in varNames:
                #getting data
                masked_data = VARs[varName][selectedPs]
                
                NaN_mask = ~np.isnan(masked_data)
                masked_data_valid = masked_data[NaN_mask]
                zLevels_valid = zLevels[NaN_mask]
                
                 # --- MAIN appending (all parcels go here) ---
                np.add.at(profileArray[varName]["profile_array"][:, 0], zLevels_valid, masked_data_valid)
                np.add.at(profileArray[varName]["profile_array"][:, 1], zLevels_valid, 1)
                np.add.at(profileArray[varName]["profile_array_squares"][:, 0], zLevels_valid, masked_data_valid**2)
                np.add.at(profileArray[varName]["profile_array_squares"][:, 1], zLevels_valid, 1)
                
                # --- LEFT subset (-1 only) ---
                if np.any(mask_left):
                    mask_left_valid = mask_left[NaN_mask]
                    np.add.at(profileArray[varName]["profile_array_left"][:, 0], zLevels_valid[mask_left_valid], masked_data_valid[mask_left_valid])
                    np.add.at(profileArray[varName]["profile_array_left"][:, 1], zLevels_valid[mask_left_valid], 1)
                    np.add.at(profileArray[varName]["profile_array_left_squares"][:, 0], zLevels_valid[mask_left_valid], masked_data_valid[mask_left_valid]**2)
                    np.add.at(profileArray[varName]["profile_array_left_squares"][:, 1], zLevels_valid[mask_left_valid], 1)

                # --- RIGHT subset (+1 only) ---
                if np.any(mask_right):
                    mask_right_valid = mask_right[NaN_mask]
                    np.add.at(profileArray[varName]["profile_array_right"][:, 0], zLevels_valid[mask_right_valid], masked_data_valid[mask_right_valid])
                    np.add.at(profileArray[varName]["profile_array_right"][:, 1], zLevels_valid[mask_right_valid], 1)
                    np.add.at(profileArray[varName]["profile_array_right_squares"][:, 0], zLevels_valid[mask_right_valid], masked_data_valid[mask_right_valid]**2)
                    np.add.at(profileArray[varName]["profile_array_right_squares"][:, 1], zLevels_valid[mask_right_valid], 1)

    return profileArraysDictionary

In [ ]:
def RunCode():
    for t in tqdm(loop_elements, desc="Processing"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        print("#" * 40,"\n",f"Processing timestep {t}/{loop_elements[-1]}")
        timeString = ModelData.timeStrings[t]
    
        #Forming Dictionary for Profile Arrays for current timestep
        trackedProfileArrays = CopyStructure(trackedArrays)
        profileArraysDictionary = InitializeProfileArrays(trackedProfileArrays,varNames)
        
        #getting variable data
        VARs = MakeDataDictionary(varNames, t)
        Z = GetSpatialData(t)
    
        #making tracked profiles
        profileArraysDictionary = MakeTrackedProfiles(trackedArrays,profileArraysDictionary,varNames,VARs,Z,t)
    
        #saving tracked profiles for current timestep
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, profileArraysDictionary, dataName, t)
    return profileArraysDictionary

In [ ]:
########################################
#RUNNING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
# running=False 

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    trackedArrays,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
                                                             Results_InputOutput_Class)
    
    #Removing After Ascent Count for SHALLOW parcels
    #Reason is there is a lot of shallow parcels
    #that hit their peak below 4 km, 
    #but stay in the cloud in turbulent eddies and later exit at much higher levels
    #and exit the cloudy updraft higher
    # for key1, subdict in trackedArrays.items():
    #     subdict['SHALLOW'][:,3]=0
    #testing without for now #*
    
    # Get Variable Names
    varNames = GetVarNames(dataName)

In [ ]:
if running:
    profileArraysDictionary = RunCode()

In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
#Z PROFILE VERSION
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
def RecombineProfiles_V1(ModelData, DataManager, tIndicies, dataName):
    """
    Combine tracked profiles across all timesteps using the first as a template.
    """
    print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

    trackedProfileArrays = None

    for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

        if t == tIndicies[0]:
            # Deep copy structure so we don’t overwrite the first timestep’s data
            trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
            continue  # move to next time step — skip accumulation for t=0

        # Add all later times to the initial one
        for key1 in profileArraysDictionary:
            for key2 in profileArraysDictionary[key1]:
                for varName in profileArraysDictionary[key1][key2]:
                    for arrayName in ["profile_array", "profile_array_squares",
                                      "profile_array_left", "profile_array_left_squares",
                                      "profile_array_right", "profile_array_right_squares"]:
                        trackedProfileArrays[key1][key2][varName][arrayName][:, 0:2] += (
                            profileArraysDictionary[key1][key2][varName][arrayName][:, 0:2]
                        )
    return trackedProfileArrays

def NanDivide(a,b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b!=0)

def RecombineProfiles_V2(ModelData, DataManager, tIndicies, dataName):
    """
    Combine tracked profiles across all timesteps using the first as a template.
    """
    print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

    targetArrays = ["profile_array", "profile_array_squares",
     "profile_array_left", "profile_array_left_squares",
     "profile_array_right", "profile_array_right_squares"]
    
    trackedProfileArrays = None
    for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

        if t == tIndicies[0]:
            # Deep copy structure so we don’t overwrite the first timestep’s data
            trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
            for key1 in trackedProfileArrays:
                for key2 in trackedProfileArrays[key1]:
                    for varName in trackedProfileArrays[key1][key2]:
                        for arrayName in targetArrays:
                            trackedProfileArrays[key1][key2][varName][arrayName][:, 0:2] = 0.0
            continue

        # Add all later times to the initial one
        for key1 in profileArraysDictionary:
            for key2 in profileArraysDictionary[key1]:
                for varName in profileArraysDictionary[key1][key2]:
                    for arrayName in targetArrays:
                        data = profileArraysDictionary[key1][key2][varName][arrayName]
                        timestepMean = NanDivide(data[:, 0],data[:, 1])
                        
                        trackedProfileArrays[key1][key2][varName][arrayName][:, 0] += timestepMean
                        trackedProfileArrays[key1][key2][varName][arrayName][:, 1] += np.ones_like(timestepMean)
                        
    return trackedProfileArrays

In [ ]:
if recombine==True:
    timeStrings_datetime = TrackedProfiles_DataLoading_CLASS.ConvertTimeStringsToDatetime(ModelData.timeStrings)
    timeSlice_all = np.arange(ModelData.Ntime)
    timeSlice_1 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,12) #late morning / pre-convective
    timeSlice_2 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,14) #CBL growth
    timeSlice_3 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,14,17) #mature convection / early decay

    ################################################################

    timeSlice_4 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,11) #SBF initiation 1
    timeSlice_5 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,11,12) #SBF initiation 2
    timeSlice_6 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,13) #CBL growth 1
    timeSlice_7 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,13,14) #CBL growth 2


    timeSlices = [timeSlice_all,
                  timeSlice_1, timeSlice_2, timeSlice_3,
                  timeSlice_4, timeSlice_5, timeSlice_6, timeSlice_7]
    timeLabels = ['combined',
                  'combined_10-12', 'combined_12-14', 'combined_14-17',
                  'combined_10-11', 'combined_11-12', 'combined_12-13', 'combined_13-14']

In [ ]:
if recombine==True:
    # for dataName in ["Variables",
    #                  "Entrainment","PROCESSED_Entrainment","PROCESSED_Entrainment_DivideMassFlux",
    #                  "W_Budgets","QV_Budgets","TH_Budgets"]:
    for dataName in ['Variables']:

        for timeSlice, timeLabel in zip(timeSlices, timeLabels): #looping through above time windows
            trackedProfileArrays = RecombineProfiles_V2(ModelData, DataManager_TrackedProfiles,tIndicies=timeSlice,dataName=dataName)
            TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, trackedProfileArrays, dataName, t=timeLabel)

In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
#TZ CONTOUR VERSION
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
def RecombineProfiles(ModelData, DataManager, tIndicies, dataName):
    """
    Combine tracked profiles across all timesteps using the first as a template.
    """
    print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

    trackedProfileArrays = None

    varNames = GetVarNames()

    arrayNames=['profile_array_right']
    for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

        if t == tIndicies[0]:
            # Deep copy structure so we don’t overwrite the first timestep’s data
            trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
            ProfileArraysDictionary_2D = InitializeProfileArraysDictionary_2D(profileArraysDictionary,
                                                                              varNames,arrayNames)
            continue

        # Add all later times to the initial one
        for key1 in ProfileArraysDictionary_2D:
            for key2 in ProfileArraysDictionary_2D[key1]:
                for varName in ProfileArraysDictionary_2D[key1][key2]:
                    for arrayName in arrayNames:
                        ProfileArraysDictionary_2D[key1][key2][varName][arrayName][t, :] += (
                            profileArraysDictionary[key1][key2][varName][arrayName][:, 0]
                        )
                        ProfileArraysDictionary_2D[key1][key2][varName][f"{arrayName}_count"][t, :] += (
                            profileArraysDictionary[key1][key2][varName][arrayName][:, 1]
                        )
    return ProfileArraysDictionary_2D

def GetVarNames():
    if dataName == "Variables":
        varNames=['QV','QCQI','W','THETA_v']
        varNames+=['THETA_e','VMF_g','VMF_c','RH_vapor','HMC']
    elif dataName == "UpdraftArea":
        varNames = ['UpdraftArea_g','UpdraftArea_c']
    elif dataName == "PROCESSED_Entrainment":
        varNames = ['PROCESSED_D_c', 'PROCESSED_D_g', 'PROCESSED_E_c', 'PROCESSED_E_g']
    elif dataName == "PROCESSED_Entrainment_DivideMassFlux":
        varNames = ['PROCESSED_D_DivideMassFlux_c', 'PROCESSED_D_DivideMassFlux_g', 'PROCESSED_E_DivideMassFlux_c', 'PROCESSED_E_DivideMassFlux_g']
    return varNames

def InitializeProfileArraysDictionary_2D(profileArraysDictionary, 
                                         varNames,arrayNames):
    newDict = {}
    Nt=ModelData.Ntime; Nz=ModelData.Nzh
    for Key1 in profileArraysDictionary:
        newDict[Key1] = {}
        for Key2 in profileArraysDictionary[Key1]:
            newDict[Key1][Key2] = {}
            for varName in varNames:
                newDict[Key1][Key2][varName] = {}
                for arrayName in arrayNames:
                    newDict[Key1][Key2][varName][arrayName] = np.zeros((Nt,Nz))
                    newDict[Key1][Key2][varName][f"{arrayName}_count"] = np.zeros((Nt,Nz))

    return newDict

In [ ]:
if recombine==True:    
    ProfileArraysDictionary_2D = RecombineProfiles(ModelData, DataManager_TrackedProfiles,tIndicies=np.arange(ModelData.Ntime),dataName=dataName)
    #SaveHere
    TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, ProfileArraysDictionary_2D, dataName, t='combined_TZContour')

In [ ]:
#########################################
#TESTING

In [ ]:
#comparing average type one and two

In [ ]:
def ProfileMean(profile): 
    """
    Input Requires Three Column Array 
    with Sum in 1st Column, Count in 2nd Column, and Index in 3rd Column
    Returns 1st and 3rd Column (removing zero rows)
    """
    #gets rid of rows that have no data
    profile_mean=profile[ (profile[:, 1] > 1)]; 
    #divides the data column by the counter column
    profile_mean=np.array([profile_mean[:, 0] / profile_mean[:, 1], profile_mean[:, 2]]).T 
    return profile_mean

def Compare(trackedProfileArrays,parcelDepth,
            axis=None,multiplier=1e3,
            xlabel='qv (g/kg)',ylabel='z (km)'):
    if axis is None:
        axis = plt.gca()

    color='green' if parcelDepth == "SHALLOW" else 'blue'
    a=trackedProfileArrays['CL'][parcelDepth]['QV']['profile_array']
    b=ProfileMean(profile=a)
    axis.plot(multiplier*b[:,0],b[:,1],linestyle='-',color=color)
    axis.set_ylim(0,6)
    axis.set_xlabel(xlabel)
    axis.set_ylabel(ylabel)
    
    a=trackedProfileArrays['nonCL'][parcelDepth]['QV']['profile_array']
    b=ProfileMean(profile=a)
    axis.plot(multiplier*b[:,0],b[:,1],linestyle='--',color=color)
    axis.set_ylim(0,6)
    axis.set_xlabel(xlabel)
    axis.set_ylabel(ylabel)

In [ ]:
def RecombineProfiles_V1(ModelData, DataManager, tIndicies, dataName):
    """
    Combine tracked profiles across all timesteps using the first as a template.
    """
    print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

    trackedProfileArrays = None

    for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

        if t == tIndicies[0]:
            # Deep copy structure so we don’t overwrite the first timestep’s data
            trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
            continue  # move to next time step — skip accumulation for t=0

        # Add all later times to the initial one
        for key1 in profileArraysDictionary:
            for key2 in profileArraysDictionary[key1]:
                for varName in profileArraysDictionary[key1][key2]:
                    for arrayName in ["profile_array", "profile_array_squares",
                                      "profile_array_left", "profile_array_left_squares",
                                      "profile_array_right", "profile_array_right_squares"]:
                        trackedProfileArrays[key1][key2][varName][arrayName][:, 0:2] += (
                            profileArraysDictionary[key1][key2][varName][arrayName][:, 0:2]
                        )
    return trackedProfileArrays

In [ ]:
def NanDivide(a,b):
    return np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b!=0)

def RecombineProfiles_V2(ModelData, DataManager, tIndicies, dataName):
    """
    Combine tracked profiles across all timesteps using the first as a template.
    """
    print(f"Recombining {ModelData.Ntime} TrackedProfiles files...\n")

    targetArrays = ["profile_array", "profile_array_squares",
     "profile_array_left", "profile_array_left_squares",
     "profile_array_right", "profile_array_right_squares"]
    
    trackedProfileArrays = None
    for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

        if t == tIndicies[0]:
            # Deep copy structure so we don’t overwrite the first timestep’s data
            trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
            for key1 in trackedProfileArrays:
                for key2 in trackedProfileArrays[key1]:
                    for varName in trackedProfileArrays[key1][key2]:
                        for arrayName in targetArrays:
                            trackedProfileArrays[key1][key2][varName][arrayName][:, 0:2] = 0.0
            continue  # move to next time step — skip accumulation for t=0

        # Add all later times to the initial one
        for key1 in profileArraysDictionary:
            for key2 in profileArraysDictionary[key1]:
                # for varName in profileArraysDictionary[key1][key2]:
                for varName in ['QV']:
                    for arrayName in targetArrays:
                        data = profileArraysDictionary[key1][key2][varName][arrayName].copy()
                        timestepMean = NanDivide(data[:, 0],data[:, 1])
                        validMask = data[:, 1]<=1
                        
                        trackedProfileArrays[key1][key2][varName][arrayName][:, 0] += timestepMean
                        trackedProfileArrays[key1][key2][varName][arrayName][validMask, 1] += 1
                        
    return trackedProfileArrays

In [ ]:
dataName='Variables'
timeSlice_all = np.arange(ModelData.Ntime)
timeSlice = timeSlice_all
timeLabel = 'combined'
# trackedProfileArrays1 = RecombineProfiles_V1(ModelData, DataManager_TrackedProfiles,tIndicies=timeSlice,dataName=dataName)
trackedProfileArrays2 = RecombineProfiles_V2(ModelData, DataManager_TrackedProfiles,tIndicies=timeSlice,dataName=dataName)

In [ ]:
fig,axes=plt.subplots(2,2,sharey=True,sharex=True)
axes=axes.flatten()
Compare(trackedProfileArrays1,parcelDepth='SHALLOW',axis=axes[0])
Compare(trackedProfileArrays2,parcelDepth='SHALLOW',axis=axes[1])

Compare(trackedProfileArrays1,parcelDepth='DEEP',axis=axes[2])
Compare(trackedProfileArrays2,parcelDepth='DEEP',axis=axes[3])